# What are GenAI guardrails?

This offline lab uses an internal HR/IT support assistant to make policy boundaries observable.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

LESSON_DIR = Path.cwd()
sys.path.insert(0, str(LESSON_DIR))
import guardrails_lab as lab

fixtures = LESSON_DIR / 'fixtures'
requests = lab.load_fixtures(fixtures / 'requests.json')
kb_data = lab.load_fixtures(fixtures / 'kb.json')
kb = [lab.Document(**item) for item in kb_data]
policies = lab.default_policies()
print(f'{len(requests)} requests, {len(kb)} documents, {len(policies)} policies')

## 1. Define the operating envelope as data

In [ ]:
for policy in policies:
    print({
        'policy_id': policy.policy_id,
        'scope': policy.scope,
        'signal': policy.signal,
        'decision': policy.decision_on_violation.value,
        'owner': policy.owner,
        'evidence': policy.evidence,
        'on_error': policy.on_error.value,
    })

## 2. Input rail: authentication, size, and scope

In [ ]:
for request in requests[:2] + [requests[12]]:
    identity = lab.Identity(**request['identity'])
    text = 'x' * 20_001 if request.get('oversize') else request['text']
    record = lab.guard_input(text, identity, policies)
    print(request['id'], record.decision.value, record.reasons)

print('Input checks cannot authorize a payroll transaction; execution must do that.')

## 3. Retrieval rail: tenant filter before context

In [ ]:
for request in (requests[2], requests[3]):
    identity = lab.Identity(**request['identity'])
    selected = [doc for doc in kb if doc.doc_id in request['retrieved_doc_ids']]
    filtered, records = lab.guard_retrieval(selected, identity, policies)
    print(request['id'], [doc.doc_id for doc in filtered])
    for record in records:
        print(record.decision.value, record.reasons, record.metadata)

## 4. Execution rail: authorization and verification

In [ ]:
for request in (requests[4], requests[6], requests[7]):
    identity = lab.Identity(**request['identity'])
    call = lab.ToolCall(**request['tool_call'])
    record = lab.guard_tool_call(identity, call, policies)
    print(request['id'], record.decision.value, record.reasons)

ledger = lab.Ledger()
receipt = lab.verify_tool_result(
    lab.ToolCall('read_ticket', {'ticket_id': 'ticket-42'}),
    {'success': True, 'operation_id': 'op-42'},
    ledger,
)
print(receipt.decision.value, ledger.entries)

## 5. Output rail: PII redaction and unsupported claims

In [ ]:
for request in (requests[9], requests[10], requests[11]):
    identity = lab.Identity(**request['identity'])
    selected = [doc for doc in kb if doc.doc_id in request.get('retrieved_doc_ids', [])]
    record = lab.guard_output(request['output_text'], selected, policies)
    print(request['id'], record.decision.value, record.reasons, record.metadata)

## 6. Fail-open versus fail-closed

In [ ]:
topic_outage = dict(lab.DEFAULT_DETECTORS)
topic_outage['topic_classifier'] = lab.failing_detector
identity = lab.Identity('user-2001', 'employee', 'bu-north', True)
open_record = lab.guard_input('What is the weather forecast?', identity, policies, topic_outage)
print('topic outage:', open_record.decision.value, open_record.reasons, open_record.metadata)

write_outage = {'authorization': lab.failing_detector}
write_request = requests[4]
write_identity = lab.Identity(**write_request['identity'])
write_call = lab.ToolCall(**write_request['tool_call'])
closed_record = lab.guard_tool_call(write_identity, write_call, policies, write_outage)
print('write outage:', closed_record.decision.value, closed_record.reasons, closed_record.metadata)

## 7. Reproduce why guardrails fail

In [ ]:
request = requests[4]
identity = lab.Identity(**request['identity'])
safe_ledger = lab.Ledger()
safe = lab.run_pipeline(request, identity, policies, kb, safe_ledger)
unsafe_ledger = lab.Ledger()
unsafe = lab.run_pipeline(request, identity, policies, kb, unsafe_ledger, misordered=True)
print('correct ordering:', safe.terminal.value, safe_ledger.entries)
print('misordered:', unsafe.terminal.value, unsafe_ledger.entries)
print('An output decision cannot undo a side effect that already happened.')

## 8. Decision records and summary table

In [ ]:
for request in requests:
    identity = lab.Identity(**request['identity'])
    result = lab.run_pipeline(request, identity, policies, kb, lab.Ledger())
    print(request['id'], request['expected_decision'], result.terminal.value, result.deciding_rail)

sample = lab.run_pipeline(requests[11], lab.Identity(**requests[11]['identity']), policies, kb, lab.Ledger())
print(json.loads(sample.records[-1].to_json()))

## Exercises

1. Add a `PolicyRecord` for an unlisted risk and predict its decision.
2. Flip one rail's `on_error` value and predict the outage result before running.
3. Move the role check into a system prompt string and show that the unauthorized write is no longer blocked.